# Evaluate DES galaxy, shear and cluster covariances with CoCoA

Compute **real-space and Fourier-space 3x2pt covariances**, keeping the
Gaussian (G), super-sample (SSC), connected non-Gaussian (cNG), and total
matrices separately. G describes products of power spectra and catalog
noise. SSC describes changes induced by density fluctuations larger than
the footprint. cNG describes the remaining connected four-point signal.

The first sections use six lens and four source bins. The final section
adds the angular cluster 6x2pt+N forecast, with three observed-redshift
bins and four richness bins. Its count crosses contain SSC only and its
cluster cNG uses the explicitly documented biased-tracer approximation.
The calculations share the physics in `cosmolike_notebook_utils.covariance`;
C++ exposes whole Gaussian matrices and individual C components. The C
loops use OpenMP and SIMDe; BLAS uses one thread.

**Forecast model:** massless neutrinos, linear galaxy bias, zero IA,
magnification and RSD, Limber spectra, a spherical-cap footprint and the
five-term halo trispectrum. SSC uses an isotropic halo response transferred
to the nonlinear power, with galaxy survey-mean subtraction. These choices
do not reproduce every approximation in a supplied survey covariance.
No likelihood data or covariance is loaded or replaced.

Follow [the covariance guide](covariance/README.md) to compile and start
this notebook. The physics decomposition follows
[Krause & Eifler, Appendix A](https://arxiv.org/abs/1601.05779) and
[Takada & Hu](https://arxiv.org/abs/1302.6994).


In [ ]:
import os
from pathlib import Path
import sys

# Set the BLAS policy before importing numerical libraries. OpenMP workers
# belong to the explicit C loops, not to nested BLAS matrix operations.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
root = Path(os.environ["ROOTDIR"])
project = root/"projects/des_cluster"
sys.path.insert(0, str(root/"external_modules/code/cosmolike_core"))
sys.path.insert(0, str(root/"external_modules/code/CAMB"))
sys.path.insert(0, str(project/"interface"))
sys.path.insert(0, str(project/"covariance"))

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib
import numpy as np
from threadpoolctl import threadpool_limits
import cosmolike_des_cluster_interface as ci
import des_cluster_covariance as survey
from cosmolike_notebook_utils import covariance as cov
from cosmolike_notebook_utils.covariance.forecast import save_forecast
from cosmolike_notebook_utils import plot_covariances as pcov

blas_limit = threadpool_limits(limits=1, user_api="blas")
ci.set_omp_threads(n=8)

# Figure appearance belongs to the notebook, not the shared plotters.
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'


## Survey inputs and accuracy

The example uses the six MagLim lens and four source distributions,
number densities and shape dispersion adopted by this project's
[synthetic-data generator](scripts/make_synthetic_data.py).
The quadrature shape dispersion 0.384666 is divided by $\sqrt{2}$
to obtain the per-component input.
These galaxy and shear blocks are part of the joint analysis described by
[DES, arXiv:2503.13631](https://arxiv.org/abs/2503.13631).
The final section adds the selected cluster population and its joint
angular covariance. The Fourier section remains galaxy/shear only.

| Measurement | Layout |
| --- | --- |
| Lens and source bins | 6 lenses, 4 sources |
| Angular bins | 20 logarithmic bins, 2.5–250 arcminutes |
| Integer Fourier bands | 15 bands, multipoles 30–4000 |
| Real-space vector | 1,000 entries |
| Fourier vector | 600 entries |

Internal spectra retain all crossed pairs, including pairs excluded from
measured rows. Inspect the printed settings before interpreting the noise.

`accuracy_boost` refines interpolation tables and multipole cutoffs from
`covariance/default.yaml`. Values 1, 2, 4 and 8 retain the base internal
refinements; the cells below compare 1 and 2. `integration_accuracy` is
independent: levels 0/1/2/3/4 select precomputed GSL rules with
96/128/256/512/1024 nodes per panel. The global boost leaves that rule
unchanged. CAMB inputs and scientific measurement bins stay fixed.
The highest tested settings provide a comparison reference, not an
automatic certificate of convergence.

The multipole and lensing-window interpolation tables retain their old
nodes when the boost doubles. New nodes split existing intervals; a
larger cutoff extends the same grid. This avoids moving the slope jumps
of the linear interpolant at each refinement. It does not guarantee
monotonic errors or replace a comparison of the highest boosts.


In [ ]:
# Base integration choices are in covariance/default.yaml.
# Each boost multiplies that file's internal refinements.
boosts = [1, 2]
settings = survey.configuration(accuracy_boost=boosts[0])
camb_tables = survey.initialize(interface=ci, settings=settings)
ci.set_omp_threads(n=8)

for name in ("area_deg2", "lens_density_arcmin2", "source_density_arcmin2",
             "sigma_e_component", "accuracy_boost"):
    print(name, settings[name])
print("Non-Gaussian multipole samples:", len(settings["ng_ell"]))


def progress(stage, elapsed):
    """Print completed integration stages without starting another calculation."""
    print(f"{elapsed:8.2f} s  {stage}")


## Real-space Gaussian, SSC and connected covariance

All angular bins of one observable are consecutive. Observable rows run
through $\xi_+$, $\xi_-$, $\gamma_t$, then $w$. Pure white noise has an
infinite harmonic tail, so the real-space calculation uses analytic pair
noise; signal and mixed noise retain the finite multipole sum.

The following loop holds the physical inputs fixed and raises only the
covariance resolution. Each returned dictionary contains `gaussian`,
`ssc`, `cng`, `total`, mean `signal`, row labels and resolved `settings`.


In [ ]:
real_results = {}
for boost in boosts:
    refined = dict(settings)
    refined.update(cov.covariance_accuracy(
        accuracy_boost=boost, **settings["accuracy_parameters"],
    ))
    print("Real-space accuracy boost", boost)
    real_results[boost] = survey.compute(
        interface=ci, settings=refined, space="real", progress=progress
    )

reference_boost = boosts[-1]
real = real_results[reference_boost]
nbin = len(real["coordinate"])
real_sizes = []
for probe in (0, 1, 2, 3):
    real_sizes.append(int(np.sum(real["rows"][:, 0] == probe))*nbin)
real_labels = [r"$\xi_+$", r"$\xi_-$", r"$\gamma_t$", r"$w$"]
pcov.plot_correlation(
    covariance=real_results[boosts[0]]["total"],
    covariance_ref=real["total"], block_sizes=real_sizes,
    block_labels=real_labels,
    labels=(f"Boost {boosts[0]}", f"Boost {reference_boost}"),
)
pcov.plot_covariance_components(
    total=real["total"],
    components={
        "Gaussian": real["gaussian"],
        "SSC": real["ssc"],
        "Connected": real["cng"],
    },
    block_sizes=real_sizes, block_labels=real_labels,
    normalization="diagonal",
)


## Fourier bandpowers

A band averages its integer multipoles with weight $2\ell+1$, their number
of harmonic modes. The Gaussian term includes finite-band white noise.
SSC and cNG use the same matter calculation as the angular forecast,
projected through band weights on both matrix axes.

There is one shear E-mode spectrum per source pair; do not duplicate it
for $\xi_+$ and $\xi_-$. Fourier means average the core spectra directly.
Real-space means include the extra source-leg factor needed to match
Cocoa's angular transformation convention. Each space applies its chosen
normalization consistently to G, SSC and cNG. Thus the examples should not
be compared as if their vectors contained the same measurements.


In [ ]:
fourier_results = {}
for boost in boosts:
    refined = dict(settings)
    refined.update(cov.covariance_accuracy(
        accuracy_boost=boost, **settings["accuracy_parameters"],
    ))
    print("Fourier accuracy boost", boost)
    fourier_results[boost] = survey.compute(
        interface=ci, settings=refined, space="fourier", progress=progress
    )

fourier = fourier_results[reference_boost]
nband = len(fourier["coordinate"])
fourier_sizes = []
for probe in (0, 2, 3):
    fourier_sizes.append(int(np.sum(fourier["rows"][:, 0] == probe))*nband)
fourier_labels = [r"$C^{EE}$", r"$C^{gE}$", r"$C^{gg}$"]
pcov.plot_correlation(
    covariance=fourier_results[boosts[0]]["total"],
    covariance_ref=fourier["total"], block_sizes=fourier_sizes,
    block_labels=fourier_labels,
    labels=(f"Boost {boosts[0]}", f"Boost {reference_boost}"),
)
pcov.plot_covariance_components(
    total=fourier["total"],
    components={
        "Gaussian": fourier["gaussian"],
        "SSC": fourier["ssc"],
        "Connected": fourier["cng"],
    },
    block_sizes=fourier_sizes, block_labels=fourier_labels,
    normalization="diagonal",
)


## Positivity and accuracy changes

A covariance must give nonnegative variance to every linear combination
of measurements. A positive diagonal alone does not establish this. The
next cells report the total matrix's smallest correlation eigenvalue,
without clipping eigenvalues or adding diagonal corrections.

Generalized eigenvalues solve $C_bv=\lambda C_{\rm ref}v$ and bound variance
ratios in every direction. The diagonal comparison measures changes in
individual error bars. The two reported change columns are fractions: 0.01
means 1%. The error-bar plots below convert those fractions to percent. Both are diagnostics; final numerical acceptance
needs stable parameter errors and marginalized Fisher Figure of Merit at
representative cosmologies. The data-vector $|\Delta\chi^2|<0.2$ condition
is not a covariance convergence test.

The correlation panels follow
[Friedrich et al., Fig. 6](https://arxiv.org/abs/2012.08568). Component maps
and histograms adapt [Barreira, Krause & Schmidt, Fig. 1](https://arxiv.org/abs/1807.04266).
The plots use the computed matrices; they retain signs and do not invent
missing components. Grey pixels mark undefined normalizations.


In [ ]:
for space, results in (("Real", real_results), ("Fourier", fourier_results)):
    reference = results[reference_boost]["total"]
    print(space, "boost | minimum correlation eigenvalue |",
          "max variance change | max error change")
    for boost in boosts:
        matrix = results[boost]["total"]
        modes = cov.covariance_modes(matrix=matrix)
        print("  Positive definite:", modes["positive_definite"])
        comparison = cov.compare_covariances(matrix=matrix, reference=reference)
        errors = np.sqrt(np.diag(matrix)/np.diag(reference))
        print(boost, modes["correlation_eigenvalues"][0],
              comparison["maximum_fractional_variance_change"],
              np.max(np.abs(errors-1)))


## Error bars for the first source bin

The full matrices above retain every bin pair. For readable error-bar
panels, select the first source auto-correlation: $\xi_+$ and $\xi_-$ in
real space, and its E-mode bands in Fourier space. Each curve shows the
change in $\sqrt{C_{ii}}$ relative to the highest tested boost.


In [ ]:
nlens = len(settings["lens_density_arcmin2"])
for space, results in (("Real", real_results), ("Fourier", fourier_results)):
    selected = results[reference_boost]
    rows = selected["rows"]
    source_rows = np.flatnonzero((rows[:, 1] == nlens) & (rows[:, 2] == nlens))
    nbin = len(selected["coordinate"])
    indices = (source_rows[:, None]*nbin+np.arange(nbin)).ravel()
    reference = selected["total"][np.ix_(indices, indices)]
    curves = {}
    for boost in boosts[:-1]:
        curves[f"Boost {boost} / {reference_boost}"] = (
            results[boost]["total"][np.ix_(indices, indices)]
        )
    labels = [r"$C^{EE}_{11}$"]
    if space == "Real":
        labels = [r"$\xi_+^{11}$", r"$\xi_-^{11}$"]
    pcov.plot_covariance_diagonal(
        theta_arcmin=selected["coordinate"], covariances=curves,
        panel_labels=labels, covariance_ref=reference,
        coordinate_label=selected["coordinate_label"],
    )


## Inspect the C++ interface and individual components

`ci.covariance_gaussian_real` and `ci.covariance_gaussian_fourier` accept
complete supplied field spectra, noise and measurement operators and
return owned NumPy matrices. Low-level functions expose the Wick products,
projection, halo moments, responses and mask variance for independent tests.
Their help text gives dimensions and units. The example below inspects
interfaces without altering the installed cosmology.


In [ ]:
help(ci.covariance_gaussian_fourier)
help(ci.covariance_project)
help(ci.covariance_halo_response)


## Save and inspect the computed matrices

The next cell writes the highest tested boost to two NumPy archives in
`covariance/`, and saves the CAMB tables separately. Each forecast archive
contains separate components, total, row map, mean signal, coordinates,
radial geometry and resolved numerical/physical settings. The paths refer
to these computed outputs, not the project's supplied data files.

For stronger refinement, extend `boosts` to `[1, 2, 4]` or `[1, 2, 4, 8]`
and rerun the notebook. Outputs at the same names are replaced. Refine
the physical model separately from its numerical integration.


In [ ]:
save_forecast(result=real, filename=project/"covariance/forecast_real.npz")
save_forecast(result=fourier, filename=project/"covariance/forecast_fourier.npz")
np.savez(file=project/"covariance/forecast_camb.npz", **camb_tables)

with np.load(project/"covariance/forecast_real.npz", allow_pickle=False) as saved:
    print("Saved arrays:", saved.files)
    print("Total covariance shape:", saved["total"].shape)


## Cluster 6x2pt + counts

The joint vector contains 2,800 two-point entries and 12 absolute counts,
in the order shear–shear, galaxy–shear, galaxy clustering, cluster–galaxy,
counts, cluster clustering, cluster lensing. The calculation retains all
internal cross-bin spectra and applies the same Y operator as the mean
model to every cluster-lensing covariance cross block.

This is a **limited-model forecast**. cNG treats cluster density as a
linearly biased matter tracer. Count cross covariance contains SSC only:
non-SSC count–spectrum terms and selected-cluster one-halo cNG corrections
are omitted. SSC includes the fixed-profile abundance response of the
cluster's own halo and the observed catalog normalization. These choices,
Limber, zero environmental selection correction and the massless model
must be assessed before inference. See the [cluster guide](covariance/README.md#joint).

This separate initialization uses the cluster adapter's recorded physical
inputs. Inspect the settings and saved omissions alongside the matrix.


In [ ]:
import des_cluster_joint_covariance as joint_survey
from cosmolike_notebook_utils.covariance.forecast_cluster import (
    save_forecast as save_joint_forecast,
)

cluster_boosts = [1, 2]
cluster_settings = joint_survey.configuration(
    accuracy_boost=cluster_boosts[0], ytransform=True,
)
cluster_camb = joint_survey.initialize(interface=ci, settings=cluster_settings)
ci.set_omp_threads(n=8)
joint_results = {}
for boost in cluster_boosts:
    refined = dict(cluster_settings)
    refined.update(cov.covariance_accuracy(
        accuracy_boost=boost, **settings["accuracy_parameters"],
    ))
    print("Joint cluster accuracy boost", boost)
    joint_results[boost] = joint_survey.compute(
        interface=ci, settings=refined, progress=progress,
    )
joint = joint_results[cluster_boosts[-1]]
print("Joint shape:", joint["total"].shape)
print("Count means:", joint["mean_counts"])
print("Omitted physics:", joint["settings"]["omitted_terms"])


### Y null modes and numerical refinement

Each final cluster-lensing bin is $Y(R_{\max})=0$ by definition. All 48
such null rows remain in the 2,812-entry output. `valid_indices` removes
only these known rows for an invertible comparison, leaving 2,764 entries.
It does not apply physical scale cuts, clip eigenvalues or repair the
matrix. Positive definiteness is necessary but does not certify convergence.

The same one-parameter boost comparison tests the coupled variance modes.
The largest tested boost is a reference for comparison, not a claim of
converged Fisher errors or of completeness of the physical model.


In [ ]:
keep = joint["valid_indices"]
reference = joint["total"][np.ix_(keep, keep)]
for boost in cluster_boosts:
    matrix = joint_results[boost]["total"][np.ix_(keep, keep)]
    modes = cov.covariance_modes(matrix=matrix)
    change = cov.compare_covariances(matrix=matrix, reference=reference)
    print(boost, "positive:", modes["positive_definite"],
          "minimum correlation eigenvalue:", modes["correlation_eigenvalues"][0],
          "maximum variance change:", change["maximum_fractional_variance_change"])

# The final cluster-lensing block has 48 rows times 19 retained angles.
# Counts occupy their own 12-entry block between cg and cc.
block_sizes = [
    400,    # 20 shear-pair rows, including xi+ and xi-, times 20 angles
    480,    # 6 galaxy bins times 4 source bins times 20 angles
    120,    # 6 galaxy auto-correlations times 20 angles
    240,    # 12 cluster-galaxy rows times 20 angles
    12,     # 3 observed redshift bins times 4 richness bins
    600,    # 3 redshift bins times 10 richness pairs times 20 angles
    48*19,  # 48 cluster-lensing rows, each with its defined null bin removed
]
block_labels = [
    "ss",
    "gs",
    "gg",
    "cg",
    "N",
    "cc",
    "cY",
]
first = joint_results[cluster_boosts[0]]["total"][np.ix_(keep, keep)]
pcov.plot_correlation(
    covariance=first, covariance_ref=reference,
    block_sizes=block_sizes, block_labels=block_labels,
    labels=(f"Cluster boost {cluster_boosts[0]}",
            f"Cluster boost {cluster_boosts[-1]}"),
)
components = {}
for name in ("gaussian", "ssc", "cng"):
    components[name] = joint[name][np.ix_(keep, keep)]
pcov.plot_covariance_components(
    total=reference, components=components,
    block_sizes=block_sizes, block_labels=block_labels,
)


### Save the joint forecast

Save the full layout, including defined Y null rows, count positions,
mean signals and model omissions. The archive includes `valid_indices`
for diagnostic selection; physical likelihood cuts remain a separate
choice. Running this cell replaces only these computed forecast files.


In [ ]:
save_joint_forecast(
    result=joint, filename=project/"covariance/forecast_cluster.npz",
)
np.savez(file=project/"covariance/forecast_cluster_camb.npz", **cluster_camb)
with np.load(project/"covariance/forecast_cluster.npz", allow_pickle=False) as saved:
    print("Saved joint arrays:", saved.files)
